In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Definir la ruta local del set v2
dir_data = Path("tiles/data/tiles-v2-2021-2024")

# 2. Cargar la metadata y los tensores originales (los que tienen el suelo urbano malo)
print("Cargando dataset base...")
df_original = pd.read_parquet(dir_data / "tiles_meta.parquet")
with np.load(dir_data / "tiles_train.npz") as data:
    # Se extrae la matriz de imágenes y la lista de bandas
    images_original = data["data"]  
    bands = data["bands"]

print(f"Metadata original: {df_original.shape[0]} registros.")
print(f"Arreglo de imágenes original: {images_original.shape}")

# 3. Crear una máscara booleana para filtrar y QUITAR el suelo urbano antiguo
# Mantenemos todo lo que NO sea la clase urbana defectuosa
mascara_mantener = df_original["clase"] != "suelo_urbano"

df_filtrado = df_original[mascara_mantener].copy()
images_filtradas = images_original[mascara_mantener.values]

print(f"\nDataset base depurado (4 clases limpias): {df_filtrado.shape[0]} muestras.")
print(f"Arreglo de imágenes filtrado: {images_filtradas.shape}")

# 4. Cargar la nueva clase urbana estricta de concreto puro
print("\nCargando nueva clase urbana estricta...")
df_urbano_estricto = pd.read_parquet(dir_data / "tiles_meta_urbano_estricto.parquet")
with np.load(dir_data / "tiles_train_urbano_estricto.npz") as data:
    images_urbano_estricto = data["data"]

# 5. Homologar el nombre de la clase para que se alinee con las constantes del modelo
# Pasamos de 'suelo_urbano_estricto' al 'suelo_urbano' estándar que espera tu red
df_urbano_estricto["clase"] = "suelo_urbano"

# 6. Concatenar tanto las tablas como las matrices de imágenes
df_final = pd.concat([df_filtrado, df_urbano_estricto], ignore_index=True)
images_final = np.concatenate([images_filtradas, images_urbano_estricto], axis=0)

print("\n=== CONSOLIDACIÓN EXITOSA ===")
print(f"Total muestras finales unificadas: {df_final.shape[0]}")
print(f"Forma del tensor final de imágenes: {images_final.shape}")

# Verificar que el balanceo perfecto de 230 muestras por clase se mantenga
print("\nConteo final por clases:")
print(df_final["clase"].value_counts())

# 7. Sobrescribir los archivos principales en tiles-v2-2021-2024
print("\nGuardando archivos unificados en disco...")
df_final.to_parquet(dir_data / "tiles_meta.parquet", index=False)
np.savez_compressed(
    dir_data / "tiles_train.npz",
    data=images_final,
    bands=bands
)

print("¡Listo! Archivos principales actualizados correctamente.")

Cargando dataset base...
Metadata original: 1150 registros.
Arreglo de imágenes original: (1150, 13, 64, 64)

Dataset base depurado (4 clases limpias): 920 muestras.
Arreglo de imágenes filtrado: (920, 13, 64, 64)

Cargando nueva clase urbana estricta...

=== CONSOLIDACIÓN EXITOSA ===
Total muestras finales unificadas: 1150
Forma del tensor final de imágenes: (1150, 13, 64, 64)

Conteo final por clases:
clase
contaminacion_alta_NO2    230
vegetacion_densa          230
ozono_anomalo             230
contaminacion_alta_SO2    230
suelo_urbano              230
Name: count, dtype: int64

Guardando archivos unificados en disco...
¡Listo! Archivos principales actualizados correctamente.
